# 08 Finalize Entity-Resolution Decisions
This notebook combines deterministic and LLM decisions, creates final match/no-match/review labels, keeps only the top-ranked candidate for the final decision table, and writes both candidate-level and final top-candidate Delta tables.

In [0]:
# Load decision inputs and configure the LLM confidence threshold.
from pyspark.sql import functions as F

decisions_table = "workspace.entity_resolution_project.company_er_decisions"
llm_table = "workspace.entity_resolution_project.company_er_llm_tiebreaker"

final_candidates_table = "workspace.entity_resolution_project.company_er_final_candidates"
final_top_table = "workspace.entity_resolution_project.company_er_final_decisions"

LLM_CONFIDENCE_THRESHOLD = 0.70

decisions = spark.table(decisions_table)

llm = (
    spark.table(llm_table)
    .select(
        "left_row_key",
        "right_row_key",
        "llm_decision",
        "llm_confidence",
        "llm_reason",
        "llm_error_message",
        "llm_response_clean"
    )
)

In [0]:
# Create final candidate decisions using deterministic and LLM outputs.
final_candidates = (
    decisions
    .join(
        llm,
        on=["left_row_key", "right_row_key"],
        how="left"
    )
    .withColumn(
        "final_decision",
        F.when(F.col("decision") == "MATCH", F.lit("MATCH"))
         .when(F.col("decision") == "NO_MATCH", F.lit("NO_MATCH"))
         .when(
             (F.col("decision") == "AMBIGUOUS") &
             (F.col("llm_decision") == "MATCH") &
             (F.col("llm_confidence") >= F.lit(LLM_CONFIDENCE_THRESHOLD)),
             F.lit("MATCH")
         )
         .when(
             (F.col("decision") == "AMBIGUOUS") &
             (F.col("llm_decision") == "NO_MATCH") &
             (F.col("llm_confidence") >= F.lit(LLM_CONFIDENCE_THRESHOLD)),
             F.lit("NO_MATCH")
         )
         .otherwise(F.lit("REVIEW"))
    )
    .withColumn(
        "final_decision_source",
        F.when(F.col("decision").isin("MATCH", "NO_MATCH"), F.lit("deterministic"))
         .when(
             (F.col("decision") == "AMBIGUOUS") &
             (F.col("llm_decision").isin("MATCH", "NO_MATCH")) &
             (F.col("llm_confidence") >= F.lit(LLM_CONFIDENCE_THRESHOLD)),
             F.lit("llm_tiebreaker")
         )
         .otherwise(F.lit("manual_review"))
    )
    .withColumn(
        "final_confidence",
        F.when(F.col("final_decision_source") == "llm_tiebreaker", F.col("llm_confidence"))
         .otherwise(F.col("top1_score"))
    )
    .withColumn(
        "needs_human_review",
        F.col("final_decision") == "REVIEW"
    )
    .withColumn(
        "resolved_company_name",
        F.when(F.col("final_decision") == "MATCH", F.col("right_company_name"))
         .otherwise(F.lit(None))
    )
    .withColumn(
        "resolved_company_key",
        F.when(F.col("final_decision") == "MATCH", F.col("right_row_key"))
         .otherwise(F.lit(None))
    )
    .withColumn(
        "final_reason",
        F.concat_ws(
            " | ",
            F.concat(F.lit("deterministic_decision="), F.col("decision")),
            F.concat(F.lit("deterministic_rule="), F.col("decision_rule")),
            F.concat(F.lit("llm_decision="), F.coalesce(F.col("llm_decision"), F.lit("none"))),
            F.concat(
                F.lit("llm_confidence="),
                F.coalesce(F.round(F.col("llm_confidence"), 3).cast("string"), F.lit("none"))
            ),
            F.concat(F.lit("llm_reason="), F.coalesce(F.col("llm_reason"), F.lit("none"))),
            F.concat(F.lit("score="), F.round(F.col("composite_score"), 3)),
            F.concat(F.lit("gap="), F.round(F.col("score_gap_top1_top2"), 3)),
            F.concat(F.lit("entropy_norm="), F.round(F.col("entropy_norm"), 3))
        )
    )
)

In [0]:
# Save final candidate-level and top-candidate decision tables.
(
    final_candidates.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(final_candidates_table)
)

final_top = final_candidates.filter(F.col("candidate_rank") == 1)

(
    final_top.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(final_top_table)
)

In [0]:
# Summarize final top-candidate decisions by source.
display(
    final_top
    .groupBy("final_decision", "final_decision_source")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
# Inspect detailed final top-candidate decisions.
display(
    final_top
    .select(
        "left_row_key",
        "left_company_name",
        "right_company_name",
        "final_decision",
        "final_decision_source",
        "final_confidence",
        "needs_human_review",
        "resolved_company_name",
        "resolved_company_key",
        "top1_score",
        "top2_score",
        "score_gap_top1_top2",
        "name_similarity",
        "semantic_similarity",
        "country_match",
        "city_match",
        "llm_decision",
        "llm_confidence",
        "llm_reason",
        "final_reason"
    )
    .orderBy("final_decision", F.desc("final_confidence"))
)

In [0]:
# Print final table row-count checks.
print("Final candidate rows:", final_candidates.count())
print("Final top decision rows:", final_top.count())
print("Unique input records:", final_top.select("left_row_key").distinct().count())